# FSL-AKE-IoD — Ablation Study

Does every design modification pay for itself? Each switch in `fslake.Config` is turned **off one at a time** and the
relevant attacks are re-run. A cell is *re-opened* when an attack that the full protocol (v2) resists succeeds, or
gets strictly worse, with the switch off.

| Switch | Modification |
|---|---|
| M1 | one-way key evolution $K\leftarrow h(K\parallel SK)$ |
| M2 | implicit TID update $TID\leftarrow h(TID\parallel SK)$ |
| M3 | dual record at the ES (v2: anchor + candidates) |
| M4 | replay cache $(TID,T_1,N_1)$ |
| M5 | masked verifier $C=K\oplus h(X_{ES}\parallel TID)$ |
| M6 | two-message flow (off = add $MSG_3=\{h(SK\parallel T_3),T_3\}$ as key confirmation) |
| F2–F4 | review fixes: cache expiry from $T_1$; candidate set; retry gap |

In [1]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "proposed", "code"))
import pandas as pd
pd.set_option("display.max_colwidth", 200)
from common import Clock, AuthError, h, xor, size_bits, reset_hash_counter, hashes
import ablation, attacks_fslake as A
T = ablation.run()

## MODIFICATION × ATTACK matrix

In [2]:
atk = [a.__name__ for a in ablation.ATTACKS]
rows = []
for label, v in T.items():
    r = {"variant": label}
    for a in atk:
        res, worse = v["results"][a]
        r[res.attack] = ("RE-OPENED: " if worse else "") + res.outcome
    c = v["cost"]; r["hash DE/ES"] = f"{c['de']}/{c['es']}"; r["msgs/bits"] = f"{c['msgs']}/{c['bits']}"
    rows.append(r)
mat = pd.DataFrame(rows).set_index("variant")
mat.T

variant,v2 (all on),M1 off: no key evolution,M2 off: static TID,M3 off: single record,M4 off: no replay cache,M5 off: plain K in ES DB,M6 off: add MSG3,F2 off: cache expiry from receive time,F3 off: one old + one cur (v1 records),F4 off: no retry gap
capture: past SKs (forward secrecy),RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED
capture: link past TIDs,PARTIAL,RE-OPENED: SUCCEEDED,RE-OPENED: SUCCEEDED,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL
anonymity: passive linking,PARTIAL,PARTIAL,RE-OPENED: SUCCEEDED,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL
stolen ES DB (no X_ES),RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED,RESISTED,RESISTED
ES compromise incl. X_ES,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL,PARTIAL
de-sync: drop one message,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED
de-sync: reordered retry (2 records),RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED
de-sync: 6 withheld retries (eviction),RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RE-OPENED: SUCCEEDED
replay MSG1 inside dT,RESISTED,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED,RESISTED,RESISTED,RESISTED
replay + clock skew (+1 s),RESISTED,RESISTED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED,RE-OPENED: SUCCEEDED,RESISTED,RESISTED


In [3]:
pd.Series({k: (', '.join(v) if v else '— nothing —') for k, v in ablation.reopened(T).items()}, name='re-opened attacks').to_frame()

,re-opened attacks
v2 (all on),— nothing —
M1 off: no key evolution,"capture: past SKs (forward secrecy), capture: link past TIDs"
M2 off: static TID,"capture: link past TIDs, anonymity: passive linking"
M3 off: single record,de-sync: drop one message
M4 off: no replay cache,"replay MSG1 inside dT, replay + clock skew (+1 s)"
M5 off: plain K in ES DB,stolen ES DB (no X_ES)
M6 off: add MSG3,— nothing —
F2 off: cache expiry from receive time,replay + clock skew (+1 s)
F3 off: one old + one cur (v1 records),"de-sync: reordered retry (2 records), de-sync: 6 withheld retries (eviction)"
F4 off: no retry gap,de-sync: 6 withheld retries (eviction)


**Output explanation.**

| Switch off | Re-opened attack | Reading |
|---|---|---|
| M1 | capture → past SKs (5/5) and past-TID linking | forward secrecy comes entirely from key evolution |
| M2 | passive linking of *all* sessions; capture links every past TID | the static pseudonym is sent on every run |
| M3 | one dropped MSG2 → permanent lock-out | a single record cannot absorb a lost message |
| M4 | in-window replay accepted (and replay with clock skew) | timestamps alone are not enough (same flaw as BAKMM-IoD A4) |
| M5 | stolen ES DB → drone impersonation | the plain key table is the verifier |
| **M6** | **nothing** | adding MSG3 re-opens no attack and **lowers** ES-compromise exposure from 3/15 to **0/15** past sessions (the ES can forget the old key on confirmation) at +288 bits and +1 hash per side. **M6 is a cost optimisation that trades away ES-side forward secrecy of the last session; it is not a security modification.** |
| F2 | replay with +1 s clock skew → lock-out | v1 bug |
| F3 | reordered retry and 6 withheld retries → lock-out | v1 bug |
| F4 | 6 withheld retries evict the drone's candidate | found while reviewing F3 |

So M1–M5 and F2–F4 are each necessary: removing any one re-opens a specific attack. **The claim that "every
modification is necessary" is false for M6**, which only buys cost.

## Cost of each variant (instrumented)

In [4]:
pd.DataFrame([dict(variant=k, DE=v['cost']['de'], ES=v['cost']['es'], messages=v['cost']['msgs'], bits=v['cost']['bits'])
              for k, v in T.items()]).set_index("variant")

,DE,ES,messages,bits
variant,,,,
v2 (all on),5,7,2,1056
M1 off: no key evolution,4,6,2,1056
M2 off: static TID,4,8,2,1056
M3 off: single record,5,7,2,1056
M4 off: no replay cache,5,7,2,1056
M5 off: plain K in ES DB,5,5,2,1056
M6 off: add MSG3,6,8,3,1344
F2 off: cache expiry from receive time,5,7,2,1056
F3 off: one old + one cur (v1 records),5,7,2,1056


**Output explanation.** Most switches do not change the cost. M1/M2 off save one hash each on both sides (though with
M2 off the ES must try two records under the same TID, so its cost rises to 8). M5 off saves the ES two hashes. M6 off
costs one hash per side and 288 bits.